# Model Training — Used Car Price Prediction

**Author:** Dinidu
**Scope:** Feature engineering integration, model comparison, hyperparameter
tuning, final model selection, and pipeline serialization.

This notebook follows the modeling approach agreed for the project:

1. Load and lightly clean the raw training data (deterministic parsing only —
   NO statistical imputation here, to avoid leakage).
2. Apply the 5 feature engineering techniques from `src/feature_engineering.py`.
3. Split 80/20 — the 20% is touched only ONCE, at the very end.
4. Build a `ColumnTransformer` where imputation/encoding/scaling are FIT
   only on the 80% training fold (fixes a leakage risk present in the
   original `src/data_preprocessing.py`, which computed medians on the
   full 6,019-row file before any split existed).
5. Compare Ridge, Random Forest, and Gradient Boosting via 5-fold CV on MAE.
6. Tune the best candidate(s) with `RandomizedSearchCV`.
7. Refit on the full 80% split, evaluate once on the held-out 20%.
8. Save the final pipeline as a single joblib artifact for FastAPI.

**Note on leakage fix:** `src/data_preprocessing.clean()` fills missing
numeric values using medians computed from the *entire* training file
before any train/test split exists. That's a data leakage risk once we
split 80/20, because the held-out 20% would have influenced its own
imputation values. This notebook therefore does NOT call the median-fill
step in `clean()` — instead it does only the deterministic, leakage-free
parts (unit extraction, invalid-value correction), and lets
`SimpleImputer` inside the `ColumnTransformer` learn medians ONLY from the
80% training fold, inside cross-validation. This will be flagged to
Kavindu so `data_preprocessing.py` can eventually be aligned, but it isn't
a blocker for this notebook.

In [ ]:
import sys
sys.path.append("..")

import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import RandomizedSearchCV, KFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from src.feature_engineering import add_features, ENGINEERED_FEATURE_COLUMNS

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## 1. Load raw data and apply deterministic (leakage-free) cleaning

This mirrors the deterministic parts of `src/data_preprocessing.py`
(unit extraction, invalid-value correction) but deliberately skips its
median-imputation step, for the leakage reason explained above.

In [ ]:
DATASET_COLUMNS = [
    "record_id", "Name", "Location", "Year", "Kilometers_Driven",
    "Fuel_Type", "Transmission", "Owner_Type", "Mileage", "Engine",
    "Power", "Seats", "New_Price", "Price",
]
NUMERIC_COLUMNS = ["Year", "Kilometers_Driven", "Mileage", "Engine", "Power", "Seats", "New_Price"]
CATEGORICAL_COLUMNS = ["Location", "Fuel_Type", "Transmission", "Owner_Type"]


def _numeric_value(value):
    if pd.isna(value):
        return float("nan")
    text = str(value).replace(",", "")
    match = re.search(r"-?\d+(?:\.\d+)?", text)
    return float(match.group()) if match else float("nan")


def load_and_lightly_clean(path: Path) -> pd.DataFrame:
    """Deterministic parsing only — NO statistical imputation (leakage-free)."""
    frame = pd.read_csv(path)
    frame = frame.iloc[:, : len(DATASET_COLUMNS)].copy()
    frame.columns = DATASET_COLUMNS

    for column in NUMERIC_COLUMNS:
        frame[column] = frame[column].map(
            lambda v: (
                _numeric_value(v) * 100
                if column == "New_Price" and isinstance(v, str) and "cr" in v.lower()
                else _numeric_value(v)
            )
        )

    # Invalid measurements -> NaN (left for the pipeline's imputer to handle later)
    frame.loc[frame["Seats"] <= 0, "Seats"] = float("nan")
    frame.loc[frame["Mileage"] <= 0, "Mileage"] = float("nan")

    frame["Price"] = frame["Price"].map(_numeric_value)
    frame = frame.dropna(subset=["Price"])  # target must be present

    for column in CATEGORICAL_COLUMNS:
        frame[column] = frame[column].fillna("Unknown").astype("string")

    return frame


raw = load_and_lightly_clean(Path("../data/raw/train-data.csv"))
print(f"Rows after dropping missing-target rows: {len(raw)}")
raw.head()

## 2. Apply feature engineering (your tested module)

Uses `add_features()` from `src/feature_engineering.py` — the same
function that will later be called by the FastAPI service, so training
and inference feature logic can never drift apart.

In [ ]:
engineered = add_features(raw)
print("New engineered columns:", ENGINEERED_FEATURE_COLUMNS)
engineered[["Year", "vehicle_age", "Kilometers_Driven", "mileage_per_year",
            "Power", "Engine", "power_per_cc", "Owner_Type", "owner_rank",
            "age_mileage_interaction"]].head()

## 3. Train/test split — 80/20, held-out set touched only once

We split BEFORE any statistical fitting (imputation, encoding, scaling)
happens, so nothing about the 20% held-out set can leak into training.

In [ ]:
TARGET = "Price"

# New_Price is 86% missing and not used as a predictor at this stage —
# excluding it avoids an unreliable, mostly-imputed feature dominating
# the model. This is a deliberate, documented decision, not an oversight.
FEATURE_COLUMNS_NUMERIC = [
    "Kilometers_Driven", "Mileage", "Engine", "Power", "Seats",
    "vehicle_age", "mileage_per_year", "power_per_cc",
    "owner_rank", "age_mileage_interaction",
]
FEATURE_COLUMNS_CATEGORICAL = ["Location", "Fuel_Type", "Transmission"]
ALL_FEATURE_COLUMNS = FEATURE_COLUMNS_NUMERIC + FEATURE_COLUMNS_CATEGORICAL

X = engineered[ALL_FEATURE_COLUMNS]
y = engineered[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)
print(f"Train: {X_train.shape}, Held-out test: {X_test.shape}")

## 4. Target transformation check

`Price` is heavily right-skewed (skew ≈ 3.34, verified earlier). We will
train models on `log1p(Price)` and compare against raw `Price` at
evaluation time — decision made empirically, not assumed.

In [ ]:
print("Raw Price skew:", y_train.skew())
print("log1p(Price) skew:", np.log1p(y_train).skew())

y_train_log = np.log1p(y_train)
y_test_log = np.log1p(y_test)  # only used for evaluating log-trained models, never for fitting

## 5. Preprocessing pipeline (ColumnTransformer)

- Numeric columns: median imputation, fit ONLY on `X_train` (fixes the
  leakage issue in `data_preprocessing.py`).
- Categorical columns: one-hot encoding with `handle_unknown="ignore"`
  (protects against unseen categories at inference time).
- Scaling: applied only inside the Ridge pipeline, not the tree models.

In [ ]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
])

categorical_transformer = Pipeline(steps=[
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer(transformers=[
    ("numeric", numeric_transformer, FEATURE_COLUMNS_NUMERIC),
    ("categorical", categorical_transformer, FEATURE_COLUMNS_CATEGORICAL),
])

# Ridge needs scaling after encoding; tree models don't.
preprocessor_scaled = ColumnTransformer(transformers=[
    ("numeric", Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]), FEATURE_COLUMNS_NUMERIC),
    ("categorical", categorical_transformer, FEATURE_COLUMNS_CATEGORICAL),
])

## 6. Candidate models + 5-fold CV comparison

Three model families: linear baseline (Ridge), bagging ensemble (Random
Forest), boosting ensemble (Gradient Boosting). Compared on
cross-validated MAE, computed on the `log1p(Price)` target since that's
expected to help the linear model most; RMSE/R² reported for context.

In [ ]:
candidates = {
    "Ridge": Pipeline(steps=[
        ("preprocessor", preprocessor_scaled),
        ("model", Ridge(random_state=RANDOM_STATE)),
    ]),
    "RandomForest": Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1)),
    ]),
    "GradientBoosting": Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", GradientBoostingRegressor(random_state=RANDOM_STATE)),
    ]),
}

cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_results = {}

for name, pipeline in candidates.items():
    fold_mae = []
    for train_idx, val_idx in cv.split(X_train):
        X_fold_train, X_fold_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_fold_train, y_fold_val = y_train_log.iloc[train_idx], y_train_log.iloc[val_idx]

        pipeline.fit(X_fold_train, y_fold_train)
        preds_log = pipeline.predict(X_fold_val)

        # Convert back to actual price scale before computing MAE,
        # since MAE in log-space isn't directly meaningful in Lakhs.
        preds = np.expm1(preds_log)
        actual = np.expm1(y_fold_val)
        fold_mae.append(mean_absolute_error(actual, preds))

    cv_results[name] = {
        "mean_mae": np.mean(fold_mae),
        "std_mae": np.std(fold_mae),
        "fold_mae": fold_mae,
    }
    print(f"{name}: mean CV-MAE = {np.mean(fold_mae):.3f} Lakh "
          f"(+/- {np.std(fold_mae):.3f})")

## 7. Hyperparameter tuning on the best candidate(s)

`RandomizedSearchCV` over a modest, meaningful search space — not
exhaustive grid search, to keep tuning time reasonable given the project
timeline. Adjust `param_distributions` based on which model(s) performed
best above.

In [ ]:
from scipy.stats import randint, uniform

# NOTE: adjust this dict to target whichever model(s) scored best in the
# CV comparison above. Shown here for both tree models as an example.
param_distributions = {
    "RandomForest": {
        "model__n_estimators": randint(100, 500),
        "model__max_depth": randint(5, 30),
        "model__min_samples_leaf": randint(1, 10),
    },
    "GradientBoosting": {
        "model__n_estimators": randint(100, 400),
        "model__learning_rate": uniform(0.01, 0.2),
        "model__max_depth": randint(2, 6),
    },
}

tuned_results = {}
for name, params in param_distributions.items():
    search = RandomizedSearchCV(
        candidates[name],
        param_distributions=params,
        n_iter=20,
        cv=cv,
        scoring="neg_mean_absolute_error",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    search.fit(X_train, y_train_log)
    tuned_results[name] = search
    print(f"{name} best CV-MAE (log-space): {-search.best_score_:.4f}")
    print(f"{name} best params: {search.best_params_}\n")

## 8. Final model selection

Selection criterion (stated explicitly, per project rubric): **lowest
cross-validated MAE on the actual price scale**, with RMSE and R² as
supporting evidence — not selected on R² alone.

Pick the best-performing tuned model based on the printed results above,
then refit it on the FULL 80% training set and evaluate ONCE on the
held-out 20% test set.

In [ ]:
# Replace "GradientBoosting" below with whichever model actually won,
# based on the printed CV-MAE comparison above.
BEST_MODEL_NAME = "GradientBoosting"
final_pipeline = tuned_results[BEST_MODEL_NAME].best_estimator_

# Refit on the FULL 80% training set (already done by RandomizedSearchCV's
# refit=True default, but explicit here for clarity).
final_pipeline.fit(X_train, y_train_log)

# Evaluate ONCE on the held-out 20% - this number goes in the report.
test_preds_log = final_pipeline.predict(X_test)
test_preds = np.expm1(test_preds_log)

final_mae = mean_absolute_error(y_test, test_preds)
final_rmse = np.sqrt(mean_squared_error(y_test, test_preds))
final_r2 = r2_score(y_test, test_preds)

print(f"FINAL HELD-OUT TEST RESULTS ({BEST_MODEL_NAME}):")
print(f"  MAE:  {final_mae:.3f} Lakh")
print(f"  RMSE: {final_rmse:.3f} Lakh")
print(f"  R^2:  {final_r2:.4f}")

## 9. Save the final pipeline

Single joblib artifact containing the full preprocessing (imputer,
encoder) + trained model, ready for FastAPI to load and call directly on
raw feature-engineered input. Note the API must still call
`add_features()` before passing data into this pipeline, since feature
engineering happens BEFORE the ColumnTransformer, not inside it.

In [ ]:
import joblib

MODELS_DIR = Path("../models")
MODELS_DIR.mkdir(exist_ok=True)

joblib.dump(final_pipeline, MODELS_DIR / "final_pipeline.joblib")

metadata = {
    "model_name": BEST_MODEL_NAME,
    "target_transform": "log1p",
    "feature_columns_numeric": FEATURE_COLUMNS_NUMERIC,
    "feature_columns_categorical": FEATURE_COLUMNS_CATEGORICAL,
    "held_out_test_mae": final_mae,
    "held_out_test_rmse": final_rmse,
    "held_out_test_r2": final_r2,
    "best_params": tuned_results[BEST_MODEL_NAME].best_params_,
}
with open(MODELS_DIR / "model_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("Saved final_pipeline.joblib and model_metadata.json")